# POLICY-BOUND DECISION SYSTEM FOR CUSTOMER SUPPORT
## Banking FAQ Assistance — API-Free Demo

This notebook runs **completely without an API key, internet connection, or external AI service**.

It demonstrates the core decision-system logic using Python rules:

- Static context
- External banking policy context
- Dynamic customer context
- Memory context
- Missing-context detection
- Conflicting-context detection
- Sensitive-information guardrails
- Policy-bound responses
- Interactive Banking FAQ Assistant

### Important
This is a **working prototype/demo of the policy-bound decision layer**. It is not a real banking service and should not be used for actual financial decisions.

## STEP 1 — Import required libraries

No installation and no API key are required.

In [ ]:
import re

print("Banking FAQ Assistant initialized successfully.")
print("Mode: API-free local policy engine")

## STEP 2 — STATIC CONTEXT
### Who the assistant is and how it should behave

In [ ]:
SYSTEM_CONTEXT_GUARDED = {
    "role": "Policy-Bound Banking Customer Support Assistant",
    "rules": [
        "Never guess missing customer information.",
        "Never invent banking policies.",
        "Ask clarification when critical information is missing.",
        "Do not make decisions when information conflicts.",
        "Never request OTP, PIN, CVV or password.",
        "Use safe official-bank guidance.",
        "Keep responses concise and professional."
    ]
}

print("STATIC CONTEXT")
print("Role:", SYSTEM_CONTEXT_GUARDED["role"])
print("\nGuardrails:")
for rule in SYSTEM_CONTEXT_GUARDED["rules"]:
    print("-", rule)

## STEP 3 — EXTERNAL CONTEXT
### Approved banking policies

This acts as the system's controlled knowledge base.

In [ ]:
BANK_POLICY = {
    "lost_card": [
        "Immediately block the card using the official banking application, internet banking or bank customer care.",
        "Never ask for PIN, CVV or OTP."
    ],

    "upi_failed": [
        "First verify the transaction status.",
        "If the amount was debited but the transaction failed, check the transaction status in the official banking application.",
        "If the amount is not automatically reversed within the applicable bank processing period, contact official bank support."
    ],

    "forgot_password": [
        "Use the official bank password-reset option.",
        "Identity verification may be required."
    ],

    "balance": [
        "Balance can be checked using official mobile banking, internet banking, ATM or branch services."
    ],

    "otp": [
        "Bank employees should never ask customers to share their OTP.",
        "The assistant must never request an OTP."
    ],

    "dispute": [
        "Transaction date, transaction status and transaction type may be required.",
        "If required information is missing, do not guess."
    ],

    "refund": [
        "A refund or reversal cannot be confirmed without transaction status and relevant transaction details."
    ]
}

print("EXTERNAL CONTEXT LOADED")
print("Number of policy areas:", len(BANK_POLICY))

## STEP 4 — MEMORY CONTEXT
### Relevant information from previous interaction

In [ ]:
MEMORY_CONTEXT = {
    "previous_issue": "Customer previously asked about a banking transaction.",
    "preferred_channel": "mobile banking",
    "verified_identity": False
}

print("MEMORY CONTEXT")
for key, value in MEMORY_CONTEXT.items():
    print(f"- {key}: {value}")

## STEP 5 — HELPER FUNCTIONS
### Detect missing information, conflicts and sensitive requests

In [ ]:
def is_unknown(value):
    if value is None:
        return True

    text = str(value).strip().lower()

    return text in {
        "", "unknown", "missing", "not provided",
        "not available", "n/a", "none"
    }


def detect_conflicts(profile):
    conflicts = []

    # Detect multiple transaction status records
    status_values = []

    for key, value in profile.items():
        if "transaction_status" in key.lower() and not is_unknown(value):
            status_values.append((key, str(value).strip().lower()))

    unique_statuses = set(value for _, value in status_values)

    if len(unique_statuses) > 1:
        conflicts.append(
            "Conflicting transaction status values: "
            + ", ".join(f"{k}={v}" for k, v in status_values)
        )

    return conflicts


def contains_sensitive_request(question):
    q = question.lower()

    sensitive_patterns = [
        r"\botp\b",
        r"one[- ]time password",
        r"\bpin\b",
        r"\bcvv\b",
        r"password"
    ]

    return any(re.search(pattern, q) for pattern in sensitive_patterns)


def contains_any(question, words):
    q = question.lower()
    return any(word in q for word in words)

print("Guardrail helper functions ready.")

## STEP 6 — POLICY-BOUND DECISION ENGINE

This is the main part of the prototype.

Instead of asking an external LLM, the local engine:
1. Checks security guardrails.
2. Checks conflicting context.
3. Checks missing required information.
4. Matches the question to an approved policy.
5. Returns only policy-supported guidance.

In [ ]:
def banking_decision_engine(user_query, user_profile):

    q = user_query.lower().strip()

    # ---------------------------------------------------------
    # 1. SENSITIVE INFORMATION GUARDRAIL
    # ---------------------------------------------------------
    if contains_sensitive_request(q):
        if contains_any(q, ["otp", "one-time password", "pin", "cvv", "password"]):
            return {
                "context_status": "Security guardrail triggered",
                "decision": "Do not request or share sensitive credentials.",
                "response": (
                    "Never share your OTP, PIN, CVV or password with anyone. "
                    "The banking assistant will not ask you for these details."
                ),
                "safe_next_step": (
                    "Use the official banking application or contact official bank support."
                )
            }

    # ---------------------------------------------------------
    # 2. CONFLICT DETECTION
    # ---------------------------------------------------------
    conflicts = detect_conflicts(user_profile)

    if conflicts:
        return {
            "context_status": "Conflicting context detected",
            "decision": "Unable to determine based on the available information.",
            "response": (
                "The transaction information contains conflicting status records. "
                "I cannot safely determine the correct transaction status."
            ),
            "safe_next_step": (
                "Verify the transaction status in the official banking application "
                "or provide the correct transaction status."
            )
        }

    # ---------------------------------------------------------
    # 3. LOST ATM CARD
    # ---------------------------------------------------------
    if contains_any(q, ["lost atm card", "lost card", "atm card lost", "card lost"]):

        return {
            "context_status": "Sufficient context",
            "decision": "Apply lost-card policy.",
            "response": (
                "If your ATM card is lost, immediately block it using the official "
                "banking application, internet banking or bank customer care."
            ),
            "safe_next_step": (
                "Block the card through an official banking channel."
            )
        }

    # ---------------------------------------------------------
    # 4. OTP SAFETY
    # ---------------------------------------------------------
    if contains_any(q, ["should i share otp", "share my otp", "give otp", "otp safe"]):

        return {
            "context_status": "Sufficient context",
            "decision": "Apply OTP safety policy.",
            "response": (
                "No. Never share your OTP with anyone. Bank employees should not "
                "ask you to share your OTP."
            ),
            "safe_next_step": (
                "Continue the transaction only through the official banking channel."
            )
        }

    # ---------------------------------------------------------
    # 5. UPI FAILED
    # ---------------------------------------------------------
    if contains_any(q, ["upi", "upi transaction"]) and contains_any(
        q, ["failed", "failure", "declined"]
    ):

        status = user_profile.get("transaction_status")
        debited = str(user_profile.get("amount_debited", "")).lower()

        if is_unknown(status):
            return {
                "context_status": "Missing context",
                "decision": "Transaction status is required.",
                "response": (
                    "Please verify whether the UPI transaction is actually marked "
                    "failed, pending or successful."
                ),
                "safe_next_step": (
                    "Check the transaction status in the official banking application."
                )
            }

        if debited in ["yes", "true"]:
            return {
                "context_status": "Sufficient context",
                "decision": "Apply failed-UPI policy.",
                "response": (
                    "If the UPI transaction failed but the amount was debited, "
                    "check the transaction status in the official banking application."
                ),
                "safe_next_step": (
                    "If the amount is not automatically reversed within the applicable "
                    "bank processing period, contact official bank support."
                )
            }

        return {
            "context_status": "Sufficient context",
            "decision": "Apply failed-UPI policy.",
            "response": (
                "First verify the transaction status in the official banking application."
            ),
            "safe_next_step": (
                "Check the transaction status and follow the bank's official guidance."
            )
        }

    # ---------------------------------------------------------
    # 6. FORGOTTEN PASSWORD
    # ---------------------------------------------------------
    if contains_any(q, ["forgot password", "forgot my password", "internet banking password"]):

        return {
            "context_status": "Sufficient context",
            "decision": "Apply password-reset policy.",
            "response": (
                "Use the official bank internet-banking password-reset option. "
                "Identity verification may be required."
            ),
            "safe_next_step": (
                "Use only the official bank website or application."
            )
        }

    # ---------------------------------------------------------
    # 7. ACCOUNT BALANCE
    # ---------------------------------------------------------
    if contains_any(q, ["account balance", "check balance", "balance"]):

        return {
            "context_status": "Sufficient context",
            "decision": "Apply account-balance policy.",
            "response": (
                "You can check your balance using official mobile banking, "
                "internet banking, an ATM or branch services."
            ),
            "safe_next_step": (
                "Use an official banking channel."
            )
        }

    # ---------------------------------------------------------
    # 8. TRANSACTION DISPUTE
    # ---------------------------------------------------------
    if contains_any(q, ["dispute", "dispute this transaction", "transaction dispute"]):

        required = ["transaction_date", "transaction_status", "transaction_type"]

        missing = [
            field for field in required
            if is_unknown(user_profile.get(field))
        ]

        if missing:
            missing_names = {
                "transaction_date": "transaction date",
                "transaction_status": "transaction status",
                "transaction_type": "transaction type"
            }

            readable = ", ".join(missing_names[field] for field in missing)

            return {
                "context_status": "Missing context",
                "decision": "Unable to determine dispute eligibility.",
                "response": (
                    "I cannot determine whether the transaction can be disputed "
                    "because required transaction information is missing."
                ),
                "safe_next_step": (
                    "Please provide or verify: " + readable +
                    ". Do not share your OTP, PIN, CVV or password."
                )
            }

        return {
            "context_status": "Sufficient context",
            "decision": "Required dispute information is available.",
            "response": (
                "The required transaction information is available for evaluating "
                "the dispute under the provided policy."
            ),
            "safe_next_step": (
                "Proceed through the bank's official dispute process."
            )
        }

    # ---------------------------------------------------------
    # 9. REFUND / REVERSAL
    # ---------------------------------------------------------
    if contains_any(q, ["refund", "money back", "reversal", "reversed"]):

        required = ["transaction_status", "transaction_type"]

        missing = [
            field for field in required
            if is_unknown(user_profile.get(field))
        ]

        if missing:
            return {
                "context_status": "Missing context",
                "decision": "Unable to determine based on the available information.",
                "response": (
                    "A refund or reversal cannot be confirmed without the "
                    "relevant transaction details."
                ),
                "safe_next_step": (
                    "Verify the transaction status and transaction details "
                    "in the official banking application."
                )
            }

    # ---------------------------------------------------------
    # 10. UNKNOWN QUERY
    # ---------------------------------------------------------
    return {
        "context_status": "Policy match unavailable",
        "decision": "Unable to determine based on the available information.",
        "response": (
            "I cannot safely answer this question using the available approved "
            "banking policies."
        ),
        "safe_next_step": (
            "Please contact official bank support or ask about one of the supported FAQ topics."
        )
    }


def print_decision(result):
    print("Context Status:")
    print(result["context_status"])

    print("\nDecision:")
    print(result["decision"])

    print("\nResponse:")
    print(result["response"])

    print("\nSafe Next Step:")
    print(result["safe_next_step"])

print("Policy-bound decision engine ready.")

# STEP 7 — SCENARIO 1
## MISSING CONTEXT

Question: **Can I dispute this transaction?**

Transaction date, status and type are missing.

In [ ]:
print("=" * 70)
print("SCENARIO 1 — MISSING CONTEXT")
print("=" * 70)

user_query_1 = "Can I dispute this transaction?"

user_profile_missing = {
    "customer_role": "account holder",
    "transaction_status": "unknown",
    "transaction_type": "unknown",
    "transaction_date": "missing"
}

print("\nDYNAMIC CONTEXT:")
print("Customer Question:", user_query_1)

for key, value in user_profile_missing.items():
    print(f"- {key}: {value}")

print("\nAI / POLICY ENGINE RESPONSE:\n")

result_1 = banking_decision_engine(
    user_query_1,
    user_profile_missing
)

print_decision(result_1)

print("\nEXPECTED BEHAVIOR:")
print("- Do NOT guess eligibility.")
print("- Detect missing transaction information.")
print("- Ask for clarification.")

# STEP 8 — SCENARIO 2
## CONFLICTING CONTEXT

Two transaction-status records disagree.

In [ ]:
print("=" * 70)
print("SCENARIO 2 — CONFLICTING CONTEXT")
print("=" * 70)

user_query_2 = "Can I get my money back for this transaction?"

user_profile_conflict = {
    "customer_role": "account holder",
    "transaction_status": "failed",
    "amount_debited": "No",
    "amount_reversed": "Yes",
    "transaction_status_record_2": "successful",
    "transaction_type": "UPI"
}

print("\nDYNAMIC CONTEXT:")
print("Customer Question:", user_query_2)

for key, value in user_profile_conflict.items():
    print(f"- {key}: {value}")

print("\nAI / POLICY ENGINE RESPONSE:\n")

result_2 = banking_decision_engine(
    user_query_2,
    user_profile_conflict
)

print_decision(result_2)

print("\nEXPECTED BEHAVIOR:")
print("- Detect conflicting transaction status.")
print("- Do not choose one status randomly.")
print("- Give a safe fallback.")

# STEP 9 — SCENARIO 3
## SUFFICIENT CONTEXT — LOST ATM CARD

In [ ]:
print("=" * 70)
print("SCENARIO 3 — SUFFICIENT CONTEXT")
print("=" * 70)

user_query_3 = "I lost my ATM card. What should I do?"

user_profile_complete = {
    "customer_role": "account holder",
    "card_status": "lost",
    "identity_verified": True
}

print("\nDYNAMIC CONTEXT:")
print("Customer Question:", user_query_3)

for key, value in user_profile_complete.items():
    print(f"- {key}: {value}")

print("\nAI / POLICY ENGINE RESPONSE:\n")

result_3 = banking_decision_engine(
    user_query_3,
    user_profile_complete
)

print_decision(result_3)

print("\nEXPECTED BEHAVIOR:")
print("- Required context is sufficient.")
print("- Apply the lost-card policy.")
print("- Recommend blocking the card through official channels.")

# STEP 10 — SCENARIO 4
## SENSITIVE INFORMATION GUARDRAIL — OTP

In [ ]:
print("=" * 70)
print("SCENARIO 4 — SENSITIVE INFORMATION")
print("=" * 70)

user_query_4 = "My bank transfer failed. Give me your OTP so you can verify it."

user_profile_security = {
    "customer_role": "account holder",
    "transaction_status": "failed",
    "identity_verified": False
}

print("\nDYNAMIC CONTEXT:")
print("Customer Question:", user_query_4)

print("\nAI / POLICY ENGINE RESPONSE:\n")

result_4 = banking_decision_engine(
    user_query_4,
    user_profile_security
)

print_decision(result_4)

print("\nEXPECTED BEHAVIOR:")
print("- Never request or expose OTP.")
print("- Tell the customer not to share OTP.")
print("- Redirect to official bank channels.")

# STEP 11 — SCENARIO 5
## FAILED UPI TRANSACTION

In [ ]:
print("=" * 70)
print("SCENARIO 5 — FAILED UPI TRANSACTION")
print("=" * 70)

user_query_5 = "My UPI payment failed and the amount was debited. What should I do?"

user_profile_upi = {
    "customer_role": "account holder",
    "transaction_type": "UPI",
    "transaction_status": "failed",
    "amount_debited": "Yes",
    "amount_reversed": "No"
}

print("\nDYNAMIC CONTEXT:")
print("Customer Question:", user_query_5)

for key, value in user_profile_upi.items():
    print(f"- {key}: {value}")

print("\nAI / POLICY ENGINE RESPONSE:\n")

result_5 = banking_decision_engine(
    user_query_5,
    user_profile_upi
)

print_decision(result_5)

print("\nEXPECTED BEHAVIOR:")
print("- Recognize the failed UPI transaction.")
print("- Recognize that the amount was debited.")
print("- Give only the approved policy guidance.")

# STEP 12 — SCENARIO 6
## FORGOTTEN INTERNET BANKING PASSWORD

In [ ]:
print("=" * 70)
print("SCENARIO 6 — FORGOTTEN PASSWORD")
print("=" * 70)

user_query_6 = "I forgot my internet banking password. What should I do?"

user_profile_password = {
    "customer_role": "account holder",
    "identity_verified": False
}

result_6 = banking_decision_engine(
    user_query_6,
    user_profile_password
)

print_decision(result_6)

# STEP 13 — INTERACTIVE BANKING FAQ ASSISTANT

This version works locally and does **not** call any API.

Type `exit` to stop.

In [ ]:
while True:

    question = input("\nCustomer: ").strip()

    if question.lower() == "exit":
        print("Banking FAQ Assistant: Session ended safely.")
        break

    # Simulated customer context
    user_profile = {
        "customer_role": "account holder",
        "transaction_status": "unknown",
        "transaction_type": "unknown",
        "transaction_date": "unknown",
        "amount_debited": "unknown",
        "amount_reversed": "unknown",
        "identity_verified": False
    }

    result = banking_decision_engine(
        question,
        user_profile
    )

    print("\nBanking FAQ Assistant:")
    print_decision(result)

# STEP 14 — COMPLETE DECISION FLOW

### Customer Question
↓  
### Static Context
Who the assistant is + guardrails  
↓  
### External Context
Approved banking policies  
↓  
### Dynamic Context
Current customer and transaction information  
↓  
### Memory Context
Relevant previous interaction  
↓  
### Security Check
OTP / PIN / CVV / password protection  
↓  
### Conflict Check
Are the supplied details contradictory?  
↓  
### Missing Context Check
Is required information available?  
↓  
### Policy Matching
Which approved banking policy applies?  
↓  
### Controlled Response
Answer / clarification / safe fallback

# STEP 15 — CONTEXT ENGINEERING SUMMARY

| Context Type | Used For |
|---|---|
| **Static Context** | Defines assistant role, behavior and guardrails |
| **External Context** | Provides approved banking policies |
| **Dynamic Context** | Provides current customer and transaction details |
| **Memory Context** | Provides relevant previous interaction information |

### Guardrail Examples

**Missing information**
→ Ask clarification  
→ Do not guess

**Conflicting information**
→ Detect conflict  
→ Do not select randomly  
→ Give safe fallback

**Sensitive information**
→ Never request OTP/PIN/CVV/password  
→ Redirect to official bank channel

**Sufficient information**
→ Match approved policy  
→ Provide controlled response

# STEP 16 — FINAL RESULT

The API-free prototype demonstrates:

✓ Policy-bound banking FAQ assistance  
✓ Static context  
✓ External policy context  
✓ Dynamic customer context  
✓ Memory context  
✓ Missing-context detection  
✓ Controlled clarification  
✓ Conflicting-context detection  
✓ Safe fallback  
✓ OTP/PIN/CVV/password guardrails  
✓ Lost-card guidance  
✓ Failed UPI guidance  
✓ Password-reset guidance  
✓ Account-balance guidance  
✓ Transaction-dispute handling  
✓ Refund/reversal handling  
✓ Interactive customer support  

## Key takeaway

A policy-bound system should **not answer every question blindly**.

It first checks:

**Context → Conflict → Missing information → Policy → Decision → Safe response**

This reduces unsupported or unsafe banking responses.

### Note
This notebook is a demonstration prototype. A production banking system would require authenticated bank systems, approved policy sources, audit logging, access controls and formal security/compliance review.